# NHL Boxscore API Explorer — Star Schema Design

**Issue #160** — Data exploration artifact for the NHL Dashboard analytics pipeline.

This notebook samples 25–50 games from the `/v1/gamecenter/{game_id}/boxscore` endpoint,
explores the full JSON response structure, and proposes a star schema for storing
per-player and per-team boxscore stats in SQLite.

## Overview

The `/v1/gamecenter/{game_id}/boxscore` endpoint returns:
- **Game metadata**: id, season, gameType, gameDate, venue, startTimeUTC, gameState
- **Team summaries**: awayTeam / homeTeam with score, shots on goal, abbreviation
- **Player stats by game**: `playerByGameStats` → forwards, defense, goalies per team
- **Game clock / period**: clock, periodDescriptor

## Setup

```bash
pip install jupyter pandas httpx
jupyter notebook nhl-dashboard/notebooks/boxscore_explorer.ipynb
```

Run all cells top-to-bottom. The batch fetch in Section 2 calls the live NHL API
for 25–50 game IDs — expect ~30–60 s total depending on network speed.

## Setup — Imports and Game ID List

Game IDs span regular season (gameType=2) and playoff (gameType=3) games from the
2025-26 season. Regular season IDs follow `2025020NNNN`; playoff IDs follow
`2025030RRGG` (RR = round, GG = game number in series).

The 50 IDs below are drawn from completed games across multiple dates and teams
to ensure the field inventory captures the full variance of the API response.

In [ ]:
import json
import time
from collections import defaultdict
from pprint import pprint

import httpx
import pandas as pd

NHL_BASE = "https://api-web.nhle.com/v1"

# 50 game IDs: 35 regular season + 15 playoff games from the 2025-26 season.
# Regular season sample — spread across October 2025 through April 2026.
REGULAR_SEASON_GAME_IDS = [
    2025020001, 2025020050, 2025020100, 2025020150, 2025020200,
    2025020300, 2025020400, 2025020500, 2025020600, 2025020700,
    2025020800, 2025020900, 2025021000, 2025021100, 2025021200,
    2025020025, 2025020075, 2025020125, 2025020175, 2025020250,
    2025020350, 2025020450, 2025020550, 2025020650, 2025020750,
    2025020850, 2025020950, 2025021050, 2025021150, 2025021250,
    2025020010, 2025020060, 2025020110, 2025020160, 2025020210,
]

# Playoff sample — Rounds 1-3 of the 2025-26 playoffs.
PLAYOFF_GAME_IDS = [
    2025030111, 2025030112, 2025030113,   # R1 S1 G1-G3
    2025030121, 2025030122, 2025030123,   # R1 S2 G1-G3
    2025030131, 2025030132,               # R1 S3 G1-G2
    2025030211, 2025030212,               # R2 S1 G1-G2
    2025030221, 2025030222,               # R2 S2 G1-G2
    2025030311, 2025030312, 2025030313,   # R3 S1 G1-G3
]

GAME_IDS = REGULAR_SEASON_GAME_IDS + PLAYOFF_GAME_IDS
print(f"Total game IDs to sample: {len(GAME_IDS)}")
print(f"  Regular season : {len(REGULAR_SEASON_GAME_IDS)}")
print(f"  Playoffs       : {len(PLAYOFF_GAME_IDS)}")

## Section 1 — API Exploration

Fetch a single boxscore and inspect the raw JSON shape.
This is the reference response that drives the field inventory in Section 3.

### Key top-level fields

| Field | Type | Description |
|---|---|---|
| `id` | int | NHL gamePk |
| `season` | int | Season (e.g. 20252026) |
| `gameType` | int | 1=preseason, 2=regular, 3=playoffs |
| `gameDate` | str | `YYYY-MM-DD` |
| `venue` | dict | `{"default": "..."}` |
| `startTimeUTC` | str | ISO 8601 UTC timestamp |
| `gameState` | str | `FUT`, `PRE`, `LIVE`, `CRIT`, `FINAL`, `OFF` |
| `awayTeam` | dict | Team summary with id, name, abbrev, score, sog |
| `homeTeam` | dict | Same as awayTeam |
| `clock` | dict | `timeRemaining`, `secondsRemaining`, `inIntermission` |
| `periodDescriptor` | dict | `number`, `periodType`, `maxRegulationPeriods` |
| `playerByGameStats` | dict | Per-player stats nested by team and position group |

In [ ]:
# Fetch a single reference game to inspect the response shape.
SAMPLE_GAME_ID = GAME_IDS[0]

resp = httpx.get(f"{NHL_BASE}/gamecenter/{SAMPLE_GAME_ID}/boxscore", timeout=15)
resp.raise_for_status()
sample = resp.json()

print(f"Game ID    : {sample.get('id')}")
print(f"Season     : {sample.get('season')}")
print(f"Game type  : {sample.get('gameType')} (2=regular, 3=playoffs)")
print(f"Game date  : {sample.get('gameDate')}")
print(f"Game state : {sample.get('gameState')}")
away = sample.get('awayTeam', {})
home = sample.get('homeTeam', {})
print(f"Match-up   : {away.get('abbrev', '?')} @ {home.get('abbrev', '?')}")
print(f"Score      : {away.get('score', '?')} - {home.get('score', '?')}")
print()
print("--- Top-level keys ---")
print(list(sample.keys()))
print()
print("--- awayTeam keys ---")
print(list(away.keys()))
print()
print("--- playerByGameStats structure ---")
pbgs = sample.get('playerByGameStats', {})
for side in ('awayTeam', 'homeTeam'):
    side_data = pbgs.get(side, {})
    print(f"  {side} keys: {list(side_data.keys())}")
    for group in ('forwards', 'defense', 'goalies'):
        players = side_data.get(group, [])
        print(f"    {group}: {len(players)} players")
        if players:
            print(f"      Player keys: {list(players[0].keys())}")

## Section 2 — Batch Fetch (25–50 Games)

Fetch all 50 game IDs and collect raw responses. Games that are not yet played
(gameState = FUT) or that return a non-200 status are skipped and counted.
The result is stored in `raw_responses` for analysis in Section 3.

In [ ]:
raw_responses = []
skipped_ids = []

for game_id in GAME_IDS:
    try:
        r = httpx.get(f"{NHL_BASE}/gamecenter/{game_id}/boxscore", timeout=15)
        if r.status_code != 200:
            skipped_ids.append((game_id, f"HTTP {r.status_code}"))
            continue
        data = r.json()
        # Skip future games — playerByGameStats will be empty
        if data.get('gameState') == 'FUT':
            skipped_ids.append((game_id, 'FUT — not yet played'))
            continue
        raw_responses.append(data)
    except Exception as e:
        skipped_ids.append((game_id, str(e)))
    # Polite rate-limiting — 50 ms between requests
    time.sleep(0.05)

print(f"Fetched successfully : {len(raw_responses)} games")
print(f"Skipped / failed     : {len(skipped_ids)} games")
if skipped_ids:
    for gid, reason in skipped_ids[:10]:
        print(f"  {gid}: {reason}")
    if len(skipped_ids) > 10:
        print(f"  ... and {len(skipped_ids) - 10} more")

## Section 3 — Field Inventory

Flatten all responses into three tidy DataFrames:
- **`df_game`** — one row per game (top-level fields + team summaries)
- **`df_skater`** — one row per (game, player) for forwards and defense
- **`df_goalie`** — one row per (game, goalie)

For each DataFrame, report: column name, dtype, null rate, and example value.
This is the primary input for the star schema design in Section 4.

In [ ]:
game_rows = []
skater_rows = []
goalie_rows = []

for resp in raw_responses:
    game_id = resp.get('id')
    away = resp.get('awayTeam', {})
    home = resp.get('homeTeam', {})
    clock = resp.get('clock', {})
    period_desc = resp.get('periodDescriptor', {})

    # --- Game-level row ---
    game_rows.append({
        'game_id'              : game_id,
        'season'               : resp.get('season'),
        'game_type'            : resp.get('gameType'),
        'game_date'            : resp.get('gameDate'),
        'venue'                : resp.get('venue', {}).get('default'),
        'start_time_utc'       : resp.get('startTimeUTC'),
        'game_state'           : resp.get('gameState'),
        'away_team_id'         : away.get('id'),
        'away_team_abbrev'     : away.get('abbrev'),
        'away_score'           : away.get('score'),
        'away_sog'             : away.get('sog'),
        'home_team_id'         : home.get('id'),
        'home_team_abbrev'     : home.get('abbrev'),
        'home_score'           : home.get('score'),
        'home_sog'             : home.get('sog'),
        'clock_time_remaining' : clock.get('timeRemaining'),
        'clock_in_intermission': clock.get('inIntermission'),
        'period_number'        : period_desc.get('number'),
        'period_type'          : period_desc.get('periodType'),
    })

    # --- Per-player rows ---
    pbgs = resp.get('playerByGameStats', {})
    for side, team_obj in [('away', away), ('home', home)]:
        team_id = team_obj.get('id')
        side_key = 'awayTeam' if side == 'away' else 'homeTeam'
        side_data = pbgs.get(side_key, {})

        for group in ('forwards', 'defense'):
            for p in side_data.get(group, []):
                skater_rows.append({
                    'game_id'               : game_id,
                    'team_id'               : team_id,
                    'side'                  : side,
                    'position_group'        : group,
                    'player_id'             : p.get('playerId'),
                    'sweater_number'        : p.get('sweaterNumber'),
                    'first_name'            : p.get('name', {}).get('firstName', {}).get('default') if isinstance(p.get('name'), dict) else None,
                    'last_name'             : p.get('name', {}).get('lastName', {}).get('default') if isinstance(p.get('name'), dict) else None,
                    'position'              : p.get('position'),
                    'goals'                 : p.get('goals'),
                    'assists'               : p.get('assists'),
                    'points'                : p.get('points'),
                    'plus_minus'            : p.get('plusMinus'),
                    'pim'                   : p.get('pim'),
                    'toi'                   : p.get('toi'),
                    'hits'                  : p.get('hits'),
                    'blocked_shots'         : p.get('blockedShots'),
                    'pp_goals'              : p.get('powerPlayGoals'),
                    'pp_points'             : p.get('powerPlayPoints'),
                    'sh_goals'              : p.get('shorthandedGoals'),
                    'faceoff_win_pct'       : p.get('faceoffWinningPctg'),
                    'giveaways'             : p.get('giveaways'),
                    'takeaways'             : p.get('takeaways'),
                    'shifts'                : p.get('shifts'),
                })

        for p in side_data.get('goalies', []):
            goalie_rows.append({
                'game_id'                      : game_id,
                'team_id'                      : team_id,
                'side'                         : side,
                'player_id'                    : p.get('playerId'),
                'sweater_number'               : p.get('sweaterNumber'),
                'first_name'                   : p.get('name', {}).get('firstName', {}).get('default') if isinstance(p.get('name'), dict) else None,
                'last_name'                    : p.get('name', {}).get('lastName', {}).get('default') if isinstance(p.get('name'), dict) else None,
                'toi'                          : p.get('toi'),
                'goals_against'                : p.get('goalsAgainst'),
                'save_shots_against'           : p.get('saveShotsAgainst'),
                'save_pct'                     : p.get('savePctg'),
                'es_shots_against'             : p.get('evenStrengthShotsAgainst'),
                'pp_shots_against'             : p.get('powerPlayShotsAgainst'),
                'sh_shots_against'             : p.get('shorthandedShotsAgainst'),
                'pim'                          : p.get('pim'),
                'starter'                      : p.get('starter'),
            })

df_game   = pd.DataFrame(game_rows)
df_skater = pd.DataFrame(skater_rows)
df_goalie = pd.DataFrame(goalie_rows)

print(f"df_game   : {df_game.shape[0]} rows × {df_game.shape[1]} cols")
print(f"df_skater : {df_skater.shape[0]} rows × {df_skater.shape[1]} cols")
print(f"df_goalie : {df_goalie.shape[0]} rows × {df_goalie.shape[1]} cols")

In [ ]:
def field_inventory(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """Build a field inventory table: column, dtype, null_rate, example value."""
    rows = []
    n = len(df)
    for col in df.columns:
        null_rate = df[col].isna().mean() if n > 0 else 1.0
        # Infer data type from non-null values
        non_null = df[col].dropna()
        inferred_type = type(non_null.iloc[0]).__name__ if len(non_null) > 0 else "unknown"
        example = non_null.iloc[0] if len(non_null) > 0 else None
        unique_count = df[col].nunique(dropna=True)
        rows.append({
            'column'       : col,
            'dtype'        : str(df[col].dtype),
            'inferred_type': inferred_type,
            'null_rate'    : f"{null_rate:.1%}",
            'unique_count' : unique_count,
            'example'      : str(example)[:60] if example is not None else 'NULL',
        })
    inv = pd.DataFrame(rows)
    print(f"\n=== Field Inventory: {label} ===")
    display(inv)
    return inv


inv_game   = field_inventory(df_game, 'df_game (game-level)')
inv_skater = field_inventory(df_skater, 'df_skater (skater stats)')
inv_goalie = field_inventory(df_goalie, 'df_goalie (goalie stats)')

In [ ]:
# High-null-rate columns (>50%) — candidates for exclusion from the initial schema
print("\n--- High-null columns in df_skater (> 50% null) ---")
high_null = inv_skater[inv_skater['null_rate'].str.rstrip('%').astype(float) > 50]
print(high_null[['column', 'null_rate']].to_string(index=False) if not high_null.empty else "  (none)")

print("\n--- Game type distribution in sample ---")
print(df_game['game_type'].value_counts().rename({2: 'Regular Season', 3: 'Playoffs'}).to_string())

print("\n--- Game state distribution in sample ---")
print(df_game['game_state'].value_counts().to_string())

print("\n--- skater position_group distribution ---")
if not df_skater.empty:
    print(df_skater['position_group'].value_counts().to_string())

print("\n--- Average skater stats per game (non-null) ---")
if not df_skater.empty:
    stat_cols = ['goals', 'assists', 'points', 'plus_minus', 'pim', 'hits', 'blocked_shots']
    available = [c for c in stat_cols if c in df_skater.columns]
    print(df_skater[available].mean().round(2).to_string())

## Section 4 — Star Schema Design

Based on the field inventory, the boxscore data maps cleanly onto a **star schema**
with two fact tables and two new dimension tables (the `game` and `team` tables
already exist in the NHL Dashboard DB).

### Proposed fact tables

| Table | Grain | Source |
|---|---|---|
| `boxscore_skater_stats` | One row per (game, player) for forwards + defense | `playerByGameStats.*.forwards` + `*.defense` |
| `boxscore_goalie_stats` | One row per (game, goalie) | `playerByGameStats.*.goalies` |

### Existing dimension tables (no migration required)

| Table | Key column | Existing model |
|---|---|---|
| `game` | `game_id` | `Game` (models.py) |
| `team` | `team_id` | `Team` (models.py via `team.team_id`) |

### New dimension table

| Table | Key column | Source field |
|---|---|---|
| `player` | `player_id` | `playerByGameStats.*.*.*.playerId` |

---

### DDL — `player` dimension

```sql
CREATE TABLE player (
    player_id      INTEGER  PRIMARY KEY,   -- API: playerId
    first_name     TEXT,                   -- API: name.firstName.default
    last_name      TEXT,                   -- API: name.lastName.default
    sweater_number INTEGER,                -- API: sweaterNumber (as of last observed game)
    position       TEXT,                   -- API: position (C/L/R/D)
    is_goalie      INTEGER  NOT NULL DEFAULT 0  -- 1 if goalie, 0 if skater
);
```

### DDL — `boxscore_skater_stats` fact table

```sql
CREATE TABLE boxscore_skater_stats (
    id                  INTEGER  PRIMARY KEY AUTOINCREMENT,
    game_id             INTEGER  NOT NULL,  -- FK → game.game_id
    player_id           INTEGER  NOT NULL,  -- FK → player.player_id
    team_id             INTEGER  NOT NULL,  -- FK → team.team_id
    side                TEXT     NOT NULL,  -- 'away' | 'home'
    position_group      TEXT     NOT NULL,  -- 'forwards' | 'defense'
    position            TEXT,               -- C / L / R / D
    goals               INTEGER,            -- API: goals
    assists             INTEGER,            -- API: assists
    points              INTEGER,            -- API: points
    plus_minus          INTEGER,            -- API: plusMinus
    pim                 INTEGER,            -- API: pim (penalty minutes)
    toi                 TEXT,               -- API: toi  ('MM:SS' string)
    hits                INTEGER,            -- API: hits
    blocked_shots       INTEGER,            -- API: blockedShots
    pp_goals            INTEGER,            -- API: powerPlayGoals
    pp_points           INTEGER,            -- API: powerPlayPoints
    sh_goals            INTEGER,            -- API: shorthandedGoals
    faceoff_win_pct     REAL,               -- API: faceoffWinningPctg (NULL for defense)
    giveaways           INTEGER,            -- API: giveaways
    takeaways           INTEGER,            -- API: takeaways
    shifts              INTEGER,            -- API: shifts
    UNIQUE (game_id, player_id)
);
```

### DDL — `boxscore_goalie_stats` fact table

```sql
CREATE TABLE boxscore_goalie_stats (
    id                      INTEGER  PRIMARY KEY AUTOINCREMENT,
    game_id                 INTEGER  NOT NULL,  -- FK → game.game_id
    player_id               INTEGER  NOT NULL,  -- FK → player.player_id
    team_id                 INTEGER  NOT NULL,  -- FK → team.team_id
    side                    TEXT     NOT NULL,  -- 'away' | 'home'
    starter                 INTEGER,            -- API: starter (1=starter, 0=backup)
    toi                     TEXT,               -- API: toi ('MM:SS')
    goals_against           INTEGER,            -- API: goalsAgainst
    save_shots_against      TEXT,               -- API: saveShotsAgainst ('saves/shots')
    save_pct                REAL,               -- API: savePctg (0.0–1.0)
    es_shots_against        INTEGER,            -- API: evenStrengthShotsAgainst
    pp_shots_against        INTEGER,            -- API: powerPlayShotsAgainst
    sh_shots_against        INTEGER,            -- API: shorthandedShotsAgainst
    pim                     INTEGER,            -- API: pim
    UNIQUE (game_id, player_id)
);
```

## Section 5 — Field Mapping Table

Maps every API field to its proposed table and column name.
Fields already captured by the existing `boxscore` table are noted as `boxscore.*`.

Use this as the implementation brief for Issue #161 (SQLAlchemy models + persist functions).

In [ ]:
# Field mapping: API response path → proposed table.column
# "→" denotes the mapping direction
FIELD_MAPPINGS = [
    # --- Game-level (already in boxscore table) ---
    {"api_path": "id",                               "proposed_table": "boxscore",              "column_name": "game_id",             "notes": "existing"},
    {"api_path": "season",                           "proposed_table": "boxscore",              "column_name": "season_id",           "notes": "existing"},
    {"api_path": "gameType",                         "proposed_table": "boxscore",              "column_name": "game_type",           "notes": "existing"},
    {"api_path": "gameDate",                         "proposed_table": "boxscore",              "column_name": "game_date",           "notes": "existing"},
    {"api_path": "venue.default",                    "proposed_table": "boxscore",              "column_name": "venue",               "notes": "existing"},
    {"api_path": "startTimeUTC",                     "proposed_table": "boxscore",              "column_name": "start_time_est",      "notes": "existing (converted to ET)"},
    {"api_path": "gameState",                        "proposed_table": "boxscore",              "column_name": "game_state",          "notes": "existing"},
    {"api_path": "awayTeam.abbrev",                  "proposed_table": "boxscore",              "column_name": "away_abbrev",         "notes": "existing"},
    {"api_path": "awayTeam.name.default",            "proposed_table": "boxscore",              "column_name": "away_name",           "notes": "existing"},
    {"api_path": "awayTeam.score",                   "proposed_table": "boxscore",              "column_name": "away_score",          "notes": "existing"},
    {"api_path": "awayTeam.sog",                     "proposed_table": "boxscore",              "column_name": "away_sog",            "notes": "existing"},
    {"api_path": "homeTeam.abbrev",                  "proposed_table": "boxscore",              "column_name": "home_abbrev",         "notes": "existing"},
    {"api_path": "homeTeam.name.default",            "proposed_table": "boxscore",              "column_name": "home_name",           "notes": "existing"},
    {"api_path": "homeTeam.score",                   "proposed_table": "boxscore",              "column_name": "home_score",          "notes": "existing"},
    {"api_path": "homeTeam.sog",                     "proposed_table": "boxscore",              "column_name": "home_sog",            "notes": "existing"},
    {"api_path": "clock.timeRemaining",              "proposed_table": "boxscore",              "column_name": "clock",               "notes": "existing"},
    {"api_path": "periodDescriptor.number",          "proposed_table": "boxscore",              "column_name": "period",              "notes": "existing (formatted)"},
    # --- Team identity (not yet captured at game level) ---
    {"api_path": "awayTeam.id",                      "proposed_table": "boxscore_skater_stats", "column_name": "team_id",             "notes": "new — maps to team.team_id"},
    {"api_path": "homeTeam.id",                      "proposed_table": "boxscore_skater_stats", "column_name": "team_id",             "notes": "new — maps to team.team_id"},
    # --- Player dimension ---
    {"api_path": "playerByGameStats.*.*.playerId",   "proposed_table": "player",                "column_name": "player_id",           "notes": "new dimension table"},
    {"api_path": "playerByGameStats.*.*.sweaterNumber", "proposed_table": "player",             "column_name": "sweater_number",      "notes": "new dimension table"},
    {"api_path": "playerByGameStats.*.*.name.firstName.default", "proposed_table": "player",   "column_name": "first_name",          "notes": "new dimension table"},
    {"api_path": "playerByGameStats.*.*.name.lastName.default",  "proposed_table": "player",   "column_name": "last_name",           "notes": "new dimension table"},
    {"api_path": "playerByGameStats.*.*.position",   "proposed_table": "player",                "column_name": "position",            "notes": "new dimension table"},
    # --- Skater fact ---
    {"api_path": "playerByGameStats.*.forwards/defense.goals",         "proposed_table": "boxscore_skater_stats", "column_name": "goals",          "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.assists",       "proposed_table": "boxscore_skater_stats", "column_name": "assists",        "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.points",        "proposed_table": "boxscore_skater_stats", "column_name": "points",         "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.plusMinus",     "proposed_table": "boxscore_skater_stats", "column_name": "plus_minus",     "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.pim",           "proposed_table": "boxscore_skater_stats", "column_name": "pim",            "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.toi",           "proposed_table": "boxscore_skater_stats", "column_name": "toi",            "notes": "new fact table (MM:SS string)"},
    {"api_path": "playerByGameStats.*.forwards/defense.hits",          "proposed_table": "boxscore_skater_stats", "column_name": "hits",           "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.blockedShots",  "proposed_table": "boxscore_skater_stats", "column_name": "blocked_shots",  "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.powerPlayGoals","proposed_table": "boxscore_skater_stats", "column_name": "pp_goals",       "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.powerPlayPoints","proposed_table": "boxscore_skater_stats","column_name": "pp_points",      "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.shorthandedGoals","proposed_table": "boxscore_skater_stats","column_name": "sh_goals",     "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.faceoffWinningPctg","proposed_table": "boxscore_skater_stats","column_name": "faceoff_win_pct","notes": "NULL for defense"},
    {"api_path": "playerByGameStats.*.forwards/defense.giveaways",     "proposed_table": "boxscore_skater_stats", "column_name": "giveaways",      "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.takeaways",     "proposed_table": "boxscore_skater_stats", "column_name": "takeaways",      "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.forwards/defense.shifts",        "proposed_table": "boxscore_skater_stats", "column_name": "shifts",         "notes": "new fact table"},
    # --- Goalie fact ---
    {"api_path": "playerByGameStats.*.goalies.toi",                    "proposed_table": "boxscore_goalie_stats", "column_name": "toi",               "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.goalies.goalsAgainst",           "proposed_table": "boxscore_goalie_stats", "column_name": "goals_against",     "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.goalies.saveShotsAgainst",       "proposed_table": "boxscore_goalie_stats", "column_name": "save_shots_against","notes": "'saves/shots' string"},
    {"api_path": "playerByGameStats.*.goalies.savePctg",               "proposed_table": "boxscore_goalie_stats", "column_name": "save_pct",          "notes": "REAL 0.0–1.0"},
    {"api_path": "playerByGameStats.*.goalies.evenStrengthShotsAgainst","proposed_table": "boxscore_goalie_stats","column_name": "es_shots_against",  "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.goalies.powerPlayShotsAgainst",  "proposed_table": "boxscore_goalie_stats", "column_name": "pp_shots_against",  "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.goalies.shorthandedShotsAgainst","proposed_table": "boxscore_goalie_stats", "column_name": "sh_shots_against",  "notes": "new fact table"},
    {"api_path": "playerByGameStats.*.goalies.starter",                "proposed_table": "boxscore_goalie_stats", "column_name": "starter",            "notes": "1=starter, 0=backup"},
    {"api_path": "playerByGameStats.*.goalies.pim",                    "proposed_table": "boxscore_goalie_stats", "column_name": "pim",                "notes": "new fact table"},
]

df_mapping = pd.DataFrame(FIELD_MAPPINGS)
print("Field mapping: API field → proposed table.column")
print(f"Total mappings: {len(df_mapping)}")
display(df_mapping.groupby('proposed_table').size().rename('field_count').sort_values(ascending=False))
print()
display(df_mapping)

## Section 6 — Trade-offs and Implementation Notes

### Design decisions recorded here

| Decision | Rationale |
|---|---|
| Two fact tables (skater + goalie) | Skaters and goalies have completely different stat columns — merging them would cause ~50% NULL rate |
| `period` as dimension data on `boxscore` | Period number is game-level metadata, not a separate dimension — no `period` dim table needed |
| `toi` stored as TEXT `MM:SS` | The API returns it as a string; converting to seconds at ingest loses readability |
| `save_pct` stored as REAL (0.0–1.0) | The API returns decimal (e.g. 0.923); multiply × 100 at display time |
| Upsert strategy: `UNIQUE(game_id, player_id)` | Re-running the ingest on a live game updates stats without duplicating rows |
| `player` dimension upserted by `player_id` | Sweater numbers change between seasons — always overwrite with latest observed value |

### What this schema does NOT capture

- `gameReports` URLs (PDF/HTML report links) — not useful for the dashboard
- `rosterSpots` (full roster including scratched players) — deferred to a later issue
- Penalty-kill and power-play team-level stats (not in this endpoint; available from play-by-play)
- Individual player ice time by strength situation (available from separate player stats endpoint)

### Implementation handoff

Open a follow-up issue for:
1. SQLAlchemy models: `Player`, `BoxscoreSkaterStats`, `BoxscoreGoalieStats`
2. Persist functions: `persist_player()`, `persist_skater_stats()`, `persist_goalie_stats()`
3. APScheduler job: `refresh_boxscore_player_stats()` — run after `refresh_boxscores()`
4. API route: `GET /api/games/<game_id>/players` — returns skater + goalie rows for a given game

Reference this notebook (Issue #160) in the implementation issue.

In [ ]:
# Final validation — confirm the sample has adequate coverage
print("=== Sample Coverage Summary ===")
print(f"Games fetched : {len(raw_responses)} of {len(GAME_IDS)} attempted")

if not df_game.empty:
    print(f"Game types    : {df_game['game_type'].value_counts().to_dict()}")
    print(f"Game states   : {df_game['game_state'].value_counts().to_dict()}")
    print(f"Date range    : {df_game['game_date'].min()} → {df_game['game_date'].max()}")

if not df_skater.empty:
    print(f"Skater rows   : {len(df_skater):,} ({df_skater['player_id'].nunique()} unique players)")

if not df_goalie.empty:
    print(f"Goalie rows   : {len(df_goalie):,} ({df_goalie['player_id'].nunique()} unique goalies)")

print()
print("Field inventory complete.")
print("Star schema proposed: boxscore_skater_stats, boxscore_goalie_stats, player")
print("See Section 4 DDL and Section 5 field mapping table for implementation details.")
print("Next step → open implementation issue for SQLAlchemy models + persist functions.")